# Statistical estimation: from model to parameters

Family 3 connects a mathematical model to observations. The numerical and probability families provide derivatives and laws; established optimization and statistics libraries do the fitting. Convexity supplies consistent problems, estimators, results and composition.

This notebook is a self-contained problem book: generated samples, analytic answers, assertions and plots. Run with the environment installed using `python -m pip install -e '.[notebooks]'`.

We cover likelihood and MLE, MAP, constraints, nonlinear/linear least squares, robust estimation, regularization, intervals, delta-method uncertainty and paired bootstrap. [Contracts and finite catalog](../docs/statistical_estimation.md).

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convexity" / "math").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from convexity.math import distributions as dist, estimation as est

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.figsize": (9, 3.5), "axes.grid": True, "grid.alpha": .2})

## 1. A distribution is a model at one parameter point

Let $Y_i \sim N(\mu,\sigma^2)$ independently. A likelihood is the same density viewed as a function of the unknown parameters:

$$\ell(\theta;y)=\sum_i \log f_\theta(y_i), \qquad \hat\theta=\arg\min_\theta[-\ell(\theta;y)].$$

We choose coordinates $(\mu,\log\sigma)$ to keep scale positive. No transformations happen implicitly. The MLE has an exact reference: sample mean and population-form sample standard deviation (`ddof=0`).

In [ ]:
sample = dist.normal(2., 1.5).sample(500, rng=rng)
model = est.likelihood(lambda t: dist.normal(t[0], np.exp(t[1])), sample)
space = est.ParameterSpace([0., 0.], names=("mu", "log_sigma"))
mle = est.maximum_likelihood(model, space)
closed_form = np.array([sample.mean(), np.log(sample.std())])
assert_allclose(mle.parameters, closed_form, atol=2e-5)
pd.DataFrame({"numerical MLE": mle.parameters, "analytic MLE": closed_form,
              "standard error": mle.standard_errors}, index=mle.names)

In [ ]:
grid = np.linspace(-3, 7, 400)
fitted_law = dist.normal(mle.parameters[0], np.exp(mle.parameters[1]))
fig, ax = plt.subplots()
ax.hist(sample, bins=30, density=True, alpha=.3, label="generated sample")
ax.plot(grid, dist.normal(2, 1.5).density(grid), label="generating density")
ax.plot(grid, fitted_law.density(grid), "--", label="fitted density")
ax.set(xlabel="observation", ylabel="density", title="A fitted law composes with family 2")
ax.legend()
plt.show()

## 2. Curvature and parameter uncertainty

For a regular interior MLE, $\widehat{\mathrm{Cov}}(\hat\theta)=(-\nabla^2\ell(\hat\theta))^{-1}$. The Hessian is of the **summed** log likelihood. We use analytic derivatives when supplied, or the numerical family's checked finite differences.

For $\sigma=\exp(\eta)$ the delta method propagates uncertainty through the Jacobian: $C_\sigma\approx J C_\theta J^T$. These are parameter intervals, not intervals for future observations.

In [ ]:
assert mle.covariance is not None
assert_allclose(mle.covariance, np.diag([sample.var()/len(sample), 1/(2*len(sample))]), atol=2e-7)
sigma = est.delta_method(mle, lambda t: np.exp(t[1]), names=("sigma",))
mean_log_interval = est.wald_interval(mle)
pd.DataFrame({"estimate": mle.parameters, "lower 95%": mean_log_interval.lower,
              "upper 95%": mean_log_interval.upper}, index=mle.names), sigma.named_parameters

## 3. MAP agrees with conjugate Gaussian algebra

With known observation SD $s$ and prior $\mu\sim N(m_0,\tau^2)$,

$$v_n=(n/s^2+1/\tau^2)^{-1},\quad m_n=v_n(\textstyle\sum_i y_i/s^2+m_0/\tau^2).$$

The posterior mode equals $m_n$. Here the local Gaussian posterior approximation is exact. `laplace_interval` labels credible intervals explicitly; it is separate from frequentist `wald_interval`.

In [ ]:
small = sample[:12]
known_scale = est.likelihood(lambda t: dist.normal(t[0], 1.5), small)
posterior = est.maximum_a_posteriori(known_scale, est.gaussian_prior(0., 1.), [0.])
posterior_variance = 1 / (len(small)/1.5**2 + 1)
posterior_mean = posterior_variance * small.sum()/1.5**2
assert_allclose(posterior.parameters, [posterior_mean], atol=1e-6)
assert_allclose(posterior.covariance, [[posterior_variance]], atol=1e-7)
posterior.parameters, est.laplace_interval(posterior)

In [ ]:
u = np.linspace(-3, 4, 400)
fig, ax = plt.subplots()
ax.plot(u, dist.normal(0, 1).density(u), label="prior")
ax.plot(u, dist.normal(posterior_mean, np.sqrt(posterior_variance)).density(u), label="posterior")
ax.axvline(small.mean(), color="gray", linestyle="--", label="sample mean")
ax.set(xlabel="mean parameter", ylabel="parameter density", title="Gaussian prior and Gaussian posterior")
ax.legend()
plt.show()

## 4. Discrete models and boundary estimates

For Poisson data, the MLE rate is the sample mean. A closed nonnegative bound includes the all-zero case. A boundary solution has no ordinary interior curvature interval: the result retains the point estimate and explains the missing covariance.

In [ ]:
counts = dist.poisson(3).sample(200, rng=rng)
count_model = est.likelihood(lambda t: dist.poisson(t[0]), counts)
rate = est.maximum_likelihood(count_model, est.ParameterSpace([1.], names=("rate",), lower=0.))
assert_allclose(rate.parameters, [counts.mean()], atol=2e-5)
boundary = est.maximum_likelihood(est.likelihood(lambda t: dist.poisson(t[0]), np.zeros(10)),
                                  est.ParameterSpace([1.], lower=0.))
assert_allclose(boundary.parameters, [0.], atol=1e-10)
assert boundary.covariance is None
rate.named_parameters, boundary.diagnostics.uncertainty_note

## 5. Nonlinear least squares with known measurement noise

For $y_i=Ae^{-kt_i}+\epsilon_i$, $\epsilon_i\sim N(0,s^2)$, minimize half the sum of standardized squared residuals. Bounds keep $A,k\geq0$. `absolute_sigma=True` means the supplied residual normalization is known; otherwise covariance is scaled by residual SSE divided by $n-p$.

In [ ]:
t = np.linspace(0, 4, 80)
y = 3*np.exp(-.7*t) + rng.normal(0, .04, len(t))
decay = est.least_squares(lambda p: (p[0]*np.exp(-p[1]*t)-y)/.04,
    est.ParameterSpace([2., .5], names=("amplitude", "decay"), lower=0.), absolute_sigma=True)
assert_allclose(decay.parameters, [3., .7], atol=.04)
fig, ax = plt.subplots()
ax.scatter(t, y, s=12, alpha=.5, label="observations")
ax.plot(t, decay.parameters[0]*np.exp(-decay.parameters[1]*t), label="fitted exponential")
ax.set(xlabel="t", ylabel="y", title="Nonlinear parameter recovery")
ax.legend()
plt.show()
decay.named_parameters, est.wald_interval(decay)

## 6. Linear estimation: OLS, precision weighting and correlated errors

Supply the design matrix explicitly, including a constant column if desired. WLS weights mean relative inverse observation variance; GLS `sigma` means relative observation covariance. They both estimate a common residual scale. HC3 instead uses a heteroskedasticity-consistent coefficient covariance.

In [ ]:
x = np.linspace(-2, 2, 100)
X = np.column_stack((np.ones(len(x)), x))
y_linear = X @ [1., 2.] + rng.normal(0, .15, len(x))
ols = est.linear_least_squares(X, y_linear, names=("intercept", "slope"))
weights = np.linspace(.5, 2., len(x))
wls = est.linear_least_squares(X, y_linear, weights=weights)
gls = est.linear_least_squares(X, y_linear, sigma=np.diag(1/weights))
assert_allclose(wls.parameters, gls.parameters)
assert_allclose(wls.covariance, gls.covariance)
hc3 = est.linear_least_squares(X, y_linear, covariance_type="HC3")
pd.DataFrame({"OLS": ols.parameters, "WLS": wls.parameters, "OLS SE": ols.standard_errors,
              "HC3 SE": hc3.standard_errors}, index=ols.names)

## 7. Robust estimation resists contamination

Ordinary squares give large residuals large influence. Huber and Tukey M-estimators use established statsmodels fitting and covariance. SciPy robust residual losses also support arbitrary nonlinear models, with an explicit residual scale; those fits require separate bootstrap or sandwich uncertainty.

In [ ]:
contaminated = y_linear.copy()
contaminated[-8:] += 15
ordinary = est.linear_least_squares(X, contaminated)
huber = est.robust_linear(X, contaminated)
tukey = est.robust_linear(X, contaminated, loss="tukey")
assert np.linalg.norm(huber.parameters-[1, 2]) < np.linalg.norm(ordinary.parameters-[1, 2])
fig, ax = plt.subplots()
ax.scatter(x, contaminated, s=12, alpha=.5, label="contaminated observations")
for label, estimate in [("OLS", ordinary), ("Huber", huber), ("Tukey", tukey)]:
    ax.plot(x, X @ estimate.parameters, label=label)
ax.set(xlabel="x", ylabel="y", title="The model is unchanged; the loss changes influence")
ax.legend()
plt.show()
pd.DataFrame({"truth": [1., 2.], "OLS": ordinary.parameters,
              "Huber": huber.parameters, "Tukey": tukey.parameters}, index=["intercept", "slope"])

## 8. One regularization convention

All three adapters minimize

$$\frac{\|y-X\theta\|^2}{2n}+\alpha\left[\rho\|\theta\|_1+\frac{1-\rho}{2}\|\theta\|^2\right].$$

Ridge uses $\rho=0$, lasso $\rho=1$, and elastic net a value in between. Every supplied coordinate is penalized, including an explicit intercept. No columns are implicitly standardized. For $X^TX/n=I$, the exact solution is soft thresholding divided by $1+\alpha(1-\rho)$.

In [ ]:
z = np.array([3., -.2, -1., 0.])
orthogonal = np.sqrt(4)*np.eye(4)
alpha = .4
fits = {}
for label, ratio in [("ridge", 0.), ("elastic net", .5), ("lasso", 1.)]:
    fitted = est.regularized_linear(orthogonal, orthogonal @ z, alpha=alpha, l1_ratio=ratio)
    exact = np.sign(z)*np.maximum(np.abs(z)-alpha*ratio, 0)/(1+alpha*(1-ratio))
    assert_allclose(fitted.parameters, exact, atol=1e-10)
    assert fitted.covariance is None
    fits[label] = fitted.parameters
pd.DataFrame({"unpenalized": z, **fits})

## 9. Parameter constraints are mathematical objects

Box bounds and linear/nonlinear constraints compose with a scalar estimation objective. Here the closest feasible point to $(.8,.7)$ on the unit simplex is $(.55,.45)$. Inference on a constrained surface requires a method specific to that geometry; an unconstrained inverse Hessian is not attached automatically.

In [ ]:
constrained = est.minimize_estimate(lambda p: np.sum((p-[.8,.7])**2),
    est.ParameterSpace([.5,.5], names=("a", "b"), lower=0., upper=1.),
    constraints=[est.LinearConstraint([[1.,1.]], lower=1., upper=1.)])
assert_allclose(constrained.parameters, [.55,.45], atol=1e-6)
constrained.named_parameters, constrained.diagnostics

## 10. Paired-row bootstrap preserves the experiment

A resample must keep each design row paired with its response. The callback can return an ordinary statistic or any converged `Estimate`. SciPy supplies resampling and percentile, basic or BCa intervals. No failed resamples are silently discarded. This row bootstrap assumes independent identically distributed observations; it is not a time-series bootstrap.

In [ ]:
boot = est.bootstrap(lambda design, response: est.linear_least_squares(design, response, names=("intercept", "slope")),
    (X, y_linear), rng=rng, method="BCa", n_resamples=399)
assert boot.draws.shape == (399, 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for j, ax in enumerate(axes):
    ax.hist(boot.draws[:,j], bins=24, alpha=.65)
    ax.axvline(ols.parameters[j], color="black", label="point estimate")
    ax.axvline(boot.interval.lower[j], color="tab:red", linestyle="--", label="95% BCa interval")
    ax.axvline(boot.interval.upper[j], color="tab:red", linestyle="--")
    ax.set(xlabel=boot.estimate.names[j], ylabel="resample count")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()
boot.interval

## 11. Add a tool without modifying Convexity

A problem defines a checked mathematical input, and an estimator is a callable returning an `Estimate`. It may use a library, an analytic identity or a user-written algorithm. No registry or inheritance is required. The problem validates result shape, names and feasibility; the estimator owns its convergence and uncertainty claims.

In [ ]:
problem = est.ObjectiveProblem(lambda p: (p[0]-3)**2, est.ParameterSpace([0.], names=("location",)))
def analytic_estimator(problem):
    return est.Estimate([3.], names=problem.space.names, objective=0.,
                        diagnostics=est.Diagnostics(backend="analytic quadratic"))
custom = est.fit(problem, analytic_estimator)
assert custom.named_parameters["location"] == 3.
custom

The completed catalog has three layers: `_backends` for library adapters, `interfaces` for checked problems/results, and `builtins` for convenient estimators. Finite-difference work reuses numerical mathematics; likelihood factories reuse probability laws. No market data or application runtime is required.

Intervals retain their assumptions: frequentist sampling uncertainty, local posterior uncertainty, and bootstrap sampling variability are distinct quantities. See the [family documentation](../docs/statistical_estimation.md) for precise input shapes, objective normalization, failure behavior and backend references.